# GroundingDINO Spatial Verification Test

Quick test notebook to verify:
1. GroundingDINO detects common objects (cat, vehicle, person, table, etc.)
2. Bounding box coordinates are correct (cx,cy,w,h → x1,y1,x2,y2)
3. Spatial geometry rules fire correctly (on/under, inside/outside, left/right)

**GPU:** T4 is fine. No API calls needed.

In [ ]:
!pip install -q "transformers>=4.50.0" torch torchvision pillow accelerate

import torch, math
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import matplotlib.pyplot as plt
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Load GroundingDINO ────────────────────────────────────
gdino_id = "IDEA-Research/grounding-dino-tiny"
print(f"Loading {gdino_id}...")
gdino_proc  = AutoProcessor.from_pretrained(gdino_id)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_id).to(DEVICE)
print("✓ GroundingDINO loaded")

GDINO_TEXT_THRESHOLD = 0.10  # lowered: 0.20 was filtering valid detections
GDINO_BOX_THRESHOLD  = 0.15  # lowered: 0.25 was filtering valid detections



In [ ]:
# ── Detection ─────────────────────────────────────────────
def run_gdino(img: Image.Image, queries: list, batch_size: int = 4) -> dict:
    """
    Run GroundingDINO to detect objects.
    Returns {query: [{box:[x1,y1,x2,y2], score, label}]}

    Uses OFFICIAL HuggingFace API:
    - text as list-of-lists: [["a cat", "a dog"]]
    - Queries sent in small batches (default 4) to avoid attention dilution
    - Articles added for better text encoder performance
    """
    if not queries:
        return {}

    # Deduplicate + lowercase
    clean_queries = list(dict.fromkeys(q.strip().lower() for q in queries if q.strip()))

    # Add articles
    article_queries = []
    article_to_query = {}
    for q in clean_queries:
        if q.split()[0] in ("a", "an", "the"):
            aq = q
        else:
            aq = f"a {q}"
        article_queries.append(aq)
        article_to_query[aq] = q

    # Run in small batches — too many queries dilutes attention
    results = {q: [] for q in clean_queries}
    for batch_start in range(0, len(article_queries), batch_size):
        batch = article_queries[batch_start : batch_start + batch_size]
        batch_map = {aq: article_to_query[aq] for aq in batch}

        inputs = gdino_proc(
            images=img,
            text=[batch],              # Official list-of-lists format
            return_tensors="pt",
        ).to(DEVICE)

        with torch.no_grad():
            outputs = gdino_model(**inputs)

        results_list = gdino_proc.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=GDINO_BOX_THRESHOLD,
            text_threshold=GDINO_TEXT_THRESHOLD,
            target_sizes=[(img.height, img.width)],
        )

        raw = results_list[0]
        boxes  = raw["boxes"].cpu().numpy()
        scores = raw["scores"].cpu().numpy()

        if "text_labels" in raw:
            labels = raw["text_labels"]
        else:
            labels = raw.get("labels", [])
            if hasattr(labels, "cpu"):
                labels = labels.cpu().tolist()

        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.tolist()
            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(img.width, int(x2)), min(img.height, int(y2))

            label_str = (label if isinstance(label, str) else str(label)).strip().lower()

            # Match label → original query (try exact first, then substring)
            matched_query = batch_map.get(label_str)
            if matched_query is None:
                stripped = re.sub(r"^(a|an|the)\s+", "", label_str)
                for aq, oq in batch_map.items():
                    if oq == stripped or oq in stripped or stripped in oq:
                        matched_query = oq
                        break
            if matched_query is None:
                for q in [article_to_query[aq] for aq in batch]:
                    if q in label_str or label_str in q:
                        matched_query = q
                        break
            if matched_query is None:
                matched_query = label_str

            if matched_query not in results:
                results[matched_query] = []

            results[matched_query].append({
                "box": [x1, y1, x2, y2],
                "score": float(score),
                "label": matched_query,
            })

    return results
# ── Geometry helpers ──────────────────────────────────────
def centroid(box):
    return ((box[0]+box[2])/2, (box[1]+box[3])/2)
def horizontal_overlap(b1, b2):
    overlap = max(0, min(b1[2], b2[2]) - max(b1[0], b2[0]))
    max_w = max(b1[2]-b1[0], b2[2]-b2[0])
    return overlap / max_w if max_w > 0 else 0
def containment_ratio(inner, outer):
    xi1, yi1 = max(inner[0], outer[0]), max(inner[1], outer[1])
    xi2, yi2 = min(inner[2], outer[2]), min(inner[3], outer[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    area = (inner[2]-inner[0]) * (inner[3]-inner[1])
    return inter / area if area > 0 else 0
def iou(b1, b2):
    xi1, yi1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    xi2, yi2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (a1+a2-inter) if (a1+a2-inter) > 0 else 0
# ── Spatial geometry verification ─────────────────────────
def verify_spatial(rel: str, subj_box: list, obj_box: list) -> dict:
    """Verify spatial relation from bboxes. Returns {verdict, evidence, correct_relation}."""
    rel_lower = rel.lower().strip()
    scx, scy = centroid(subj_box)
    ocx, ocy = centroid(obj_box)
    h_ov = horizontal_overlap(subj_box, obj_box)
    contain = containment_ratio(subj_box, obj_box)
    verdict, evidence, correct = None, "", None
    if rel_lower in ["left of", "to the left of"]:
        verdict = scx < ocx
        evidence = f"subj_cx={scx:.0f} {'<' if verdict else '>='} obj_cx={ocx:.0f}"
        if not verdict: correct = "to the right of"
    elif rel_lower in ["right of", "to the right of"]:
        verdict = scx > ocx
        evidence = f"subj_cx={scx:.0f} {'>' if verdict else '<='} obj_cx={ocx:.0f}"
        if not verdict: correct = "to the left of"
    elif rel_lower in ["above"]:
        verdict = scy < ocy
        evidence = f"subj_cy={scy:.0f} {'<' if verdict else '>='} obj_cy={ocy:.0f}"
        if not verdict: correct = "below"
    elif rel_lower in ["below", "beneath"]:
        verdict = scy > ocy
        evidence = f"subj_cy={scy:.0f} {'>' if verdict else '<='} obj_cy={ocy:.0f}"
        if not verdict: correct = "above"
    elif rel_lower in ["on", "on top of"]:
        verdict = scy < ocy and h_ov > 0.3
        evidence = f"subj_cy={scy:.0f} vs obj_cy={ocy:.0f}, h_overlap={h_ov:.2f}"
        if not verdict: correct = "under" if scy >= ocy else "not aligned"
    elif rel_lower in ["under", "underneath"]:
        verdict = scy > ocy and h_ov > 0.3
        evidence = f"subj_cy={scy:.0f} vs obj_cy={ocy:.0f}, h_overlap={h_ov:.2f}"
        if not verdict: correct = "on top of"
    elif rel_lower in ["inside", "in", "within", "inside of"]:
        verdict = contain > 0.6
        evidence = f"containment={contain*100:.0f}% (need >60%)"
        if not verdict: correct = "outside" if contain < 0.2 else "partially inside"
    elif rel_lower in ["outside", "outside of"]:
        verdict = contain < 0.2
        evidence = f"containment={contain*100:.0f}% (need <20% for outside)"
        if not verdict: correct = "inside"
    elif rel_lower in ["next to", "beside", "near", "close to"]:
        img_diag = math.sqrt(max(subj_box[2], obj_box[2])**2 + max(subj_box[3], obj_box[3])**2)
        dist = math.sqrt((scx-ocx)**2 + (scy-ocy)**2)
        rel_dist = dist / img_diag if img_diag > 0 else 1.0
        verdict = rel_dist < 0.3
        evidence = f"rel_dist={rel_dist:.2f} (threshold 0.3)"
        if not verdict: correct = "far from"
    elif rel_lower in ["far from", "away from"]:
        img_diag = math.sqrt(max(subj_box[2], obj_box[2])**2 + max(subj_box[3], obj_box[3])**2)
        dist = math.sqrt((scx-ocx)**2 + (scy-ocy)**2)
        rel_dist = dist / img_diag if img_diag > 0 else 1.0
        verdict = rel_dist >= 0.3
        evidence = f"rel_dist={rel_dist:.2f} (threshold 0.3)"
        if not verdict: correct = "near"
    else:
        evidence = f"No geometry rule for: {rel_lower}"
    return {"verdict": verdict, "evidence": evidence, "correct_relation": correct}
# ── Visualization ─────────────────────────────────────────
COLORS = ["#FF0000", "#00CC00", "#0066FF", "#FF9900", "#CC00FF", "#00CCCC", "#FF3399"]
def draw_detections(img, detections_dict):
    """Draw bounding boxes on image. Returns annotated PIL image."""
    draw_img = img.copy()
    draw = ImageDraw.Draw(draw_img)
    ci = 0
    for query, dets in detections_dict.items():
        color = COLORS[ci % len(COLORS)]
        ci += 1
        for d in dets:
            x1, y1, x2, y2 = d["box"]
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            label = f"{query} ({d['score']:.2f})"
            draw.text((x1+2, max(0, y1-12)), label, fill=color)
    return draw_img
print("✓ All functions defined (using correct GroundingDINO API)")




In [ ]:
# ═══════════════════════════════════════════════════════════
# DIAGNOSTIC: Verify GroundingDINO works on a KNOWN image
# Uses official HuggingFace example: COCO val2017 cats image
# Expected: detects 2 cats and 1 remote control
# ═══════════════════════════════════════════════════════════
import requests
from io import BytesIO

print("="*60)
print("DIAGNOSTIC: Testing GroundingDINO on known image")
print("="*60)

# Official HuggingFace example image - 2 cats + remote control
DIAG_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
try:
    diag_img = Image.open(BytesIO(requests.get(DIAG_URL, timeout=10).content)).convert("RGB")
    print(f"Image: {diag_img.size[0]}x{diag_img.size[1]}px  ({DIAG_URL.split('/')[-1]})")
except Exception as e:
    print(f"Could not download image: {e}")
    print("Using a blank test image instead")
    import numpy as np
    diag_img = Image.fromarray(np.zeros((480, 640, 3), dtype=np.uint8))

# ── Test A: Minimal query (official example) ─────────────
print("\n── Test A: 2 queries (a cat, a remote control) ─────────")
DIAG_QUERIES = ["a cat", "a remote control"]

inputs_d = gdino_proc(
    images=diag_img,
    text=[DIAG_QUERIES],       # Official list-of-lists format
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    out_d = gdino_model(**inputs_d)

# Show raw scores BEFORE threshold (so we can see if model detects but threshold filters)
raw_scores = out_d.logits.sigmoid().max(dim=-1).values[0].cpu().numpy()
print(f"  Raw detection scores (top-10 before threshold):")
top10_idx = raw_scores.argsort()[::-1][:10]
for idx in top10_idx:
    print(f"    query_slot {idx}: score={raw_scores[idx]:.4f}")

# Post-process at THREE different thresholds to see what changes
for thresh in [0.10, 0.20, 0.30, 0.40]:
    res = gdino_proc.post_process_grounded_object_detection(
        out_d,
        inputs_d.input_ids,
        threshold=thresh,
        text_threshold=thresh * 0.7,
        target_sizes=[(diag_img.height, diag_img.width)],
    )
    r = res[0]
    tl = r.get('text_labels', r.get('labels', []))
    print(f"  threshold={thresh:.2f}: {len(r['boxes'])} detections → labels={list(tl)[:5]}")

# ── Test B: Our run_gdino function ───────────────────────
print("\n── Test B: run_gdino([cat, remote control]) ─────────────")
result_b = run_gdino(diag_img, ["cat", "remote control"])
for q, dets in result_b.items():
    print(f"  '{q}': {len(dets)} detections", [f"score={d['score']:.3f}" for d in dets[:3]])

# ── Test C: Single entity at a time ─────────────────────
print("\n── Test C: Single queries one at a time ────────────────")
for single_q in ["cat", "remote control", "dog", "couch"]:
    r_single = run_gdino(diag_img, [single_q])
    found = r_single.get(single_q, [])
    print(f"  '{single_q}': {len(found)} detections", [f"{d['score']:.3f}" for d in found[:3]])

# ── Visualize ────────────────────────────────────────────
all_found = {q: d for q, d in result_b.items() if d}
if all_found:
    ann = draw_detections(diag_img, all_found)
    plt.figure(figsize=(10, 6))
    plt.imshow(ann)
    plt.title(f"Diagnostic: {DIAG_QUERIES} → found {list(all_found.keys())}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print("✓ Visualization shown")
else:
    print("⚠️  Nothing detected — check raw scores above; threshold may be too high")

print("\n" + "="*60)
print("If Test A shows 0 detections at threshold=0.10, GDINO is broken.")
print("If it shows detections at 0.10 but not 0.25, just lower thresholds.")
print("If Test B shows 0 but Test A shows some, label matching is broken.")
print("="*60)


In [ ]:
# ── Use COCO images from Drive ────────────────────────────
from google.colab import drive
import os, zipfile, random

drive.mount("/content/drive")
print("✓ Drive mounted")

# Path to your COCO zips
COCO_ZIPS_DRIVE = "/content/drive/MyDrive/RelCheck_Data/coco_zips"
COCO_IMG_ZIP = f"{COCO_ZIPS_DRIVE}/val2014.zip"

TEST_DIR = "/content/test_images"
os.makedirs(TEST_DIR, exist_ok=True)

# Extract a sample of COCO images
if os.path.exists(COCO_IMG_ZIP):
    print(f"Extracting COCO images from {COCO_IMG_ZIP}...")
    with zipfile.ZipFile(COCO_IMG_ZIP, 'r') as z:
        all_files = [f for f in z.namelist() if f.endswith('.jpg')]
        # Get 5 random images
        sample = random.sample(all_files, min(5, len(all_files)))
        for fname in sample:
            z.extract(fname, TEST_DIR)
            # Flatten directory structure
            src = f"{TEST_DIR}/{fname}"
            dst = f"{TEST_DIR}/{os.path.basename(fname)}"
            if src != dst:
                os.rename(src, dst)
            print(f"  ✓ {os.path.basename(fname)}")
    
    # Get list of extracted images
    extracted = [f for f in os.listdir(TEST_DIR) if f.endswith('.jpg')]
    print(f"\n✓ {len(extracted)} COCO images ready:")
    for f in extracted[:5]:
        print(f"  {f}")
else:
    print(f"⚠️  COCO zip not found at {COCO_IMG_ZIP}")
    print("Make sure val2014.zip is uploaded to your Drive.")


In [ ]:
# ── Test 1: Object Detection ──────────────────────────────
# Verify GroundingDINO can find common objects in COCO images

print("=" * 70)
print("TEST 1: Object Detection on COCO Images")
print("=" * 70)

import os
from PIL import Image

# Get all images from TEST_DIR
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.jpg')])[:5]
all_detections = {}

# Object queries to test
COMMON_OBJECTS = [
    "person", "car", "dog", "cat", "bicycle", "motorcycle",
    "table", "chair", "cup", "bottle", "backpack", "handbag",
    "building", "tree", "grass", "road", "street", "sky"
]

for fname in test_files:
    fpath = f"{TEST_DIR}/{fname}"
    img = Image.open(fpath).convert("RGB")
    print(f"\n📷 {fname} ({img.size[0]}x{img.size[1]})")

    dets = run_gdino(img, COMMON_OBJECTS)
    found = {q: d for q, d in dets.items() if d}
    all_detections[fname] = {"img": img, "dets": dets, "found": found}

    if found:
        print(f"   Detected {len(found)} object types:")
        for q, d_list in list(found.items())[:8]:  # Show first 8
            for d in d_list[:2]:  # Show first 2 detections per type
                x1,y1,x2,y2 = d["box"]
                w, h = x2-x1, y2-y1
                print(f"     • {q:12s} {w:3d}x{h:3d}px  score={d['score']:.3f}")
    else:
        print("   ✗ No objects detected (queries may not match image)")

# Show annotated images
if all_detections:
    fig, axes = plt.subplots(1, len(all_detections), figsize=(20, 4))
    for ax, (fname, data) in zip(axes if len(all_detections) > 1 else [axes], 
                                  all_detections.items()):
        annotated = draw_detections(data["img"], data["found"])
        ax.imshow(annotated)
        ax.set_title(fname.replace('.jpg', ''), fontsize=9)
        ax.axis("off")
    plt.suptitle("GroundingDINO Detections on COCO Images", fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"\n✓ Displayed {len(all_detections)} annotated images")


In [ ]:
# ── Test 2: Spatial Geometry Rules ────────────────────────
# For each image, pick detected object pairs and test ALL relations

print("=" * 70)
print("TEST 2: Spatial Geometry Verification")
print("=" * 70)

RELATIONS_TO_TEST = [
    "on", "on top of", "under", "underneath",
    "left of", "right of", "above", "below",
    "inside", "in", "outside",
    "next to", "near", "far from",
]

total_tests = 0
for name, data in all_detections.items():
    found = data["found"]
    labels = list(found.keys())
    if len(labels) < 2:
        print(f"\n📷 {name}: <2 objects detected, skipping")
        continue

    print(f"\n📷 {name} — detected: {', '.join(labels)}")

    # Test first pair of detected objects
    subj_label = labels[0]
    obj_label  = labels[1]
    subj_box = found[subj_label][0]["box"]
    obj_box  = found[obj_label][0]["box"]

    print(f"  Testing: {subj_label} [box={subj_box}] vs {obj_label} [box={obj_box}]")
    print(f"  {'Relation':<18s} {'Verdict':<12s} {'Evidence'}")
    print(f"  {'-'*18} {'-'*12} {'-'*40}")

    for rel in RELATIONS_TO_TEST:
        result = verify_spatial(rel, subj_box, obj_box)
        v = result["verdict"]
        tag = "✓ TRUE" if v == True else ("✗ FALSE" if v == False else "? NONE")
        print(f"  {rel:<18s} {tag:<12s} {result['evidence'][:55]}")
        total_tests += 1

print(f"\n{'='*70}")
print(f"Ran {total_tests} geometry checks across {len(all_detections)} images")
print(f"{'='*70}")


In [ ]:
# ── Test 3: Corruption Detection Simulation ───────────────
# Simulate the exact scenario from the synthetic test:
# Given a TRUE relation and a CORRUPTED relation, can geometry tell them apart?

print("=" * 70)
print("TEST 3: Can geometry detect corrupted relations?")
print("=" * 70)

CORRUPTION_TESTS = [
    # (subject_query, object_query, true_relation, corrupted_relation)
    ("cat",    "couch",   "on top of",  "under"),
    ("cat",    "couch",   "on top of",  "inside"),
    ("cat",    "laptop",  "on",         "under"),
    ("cat",    "laptop",  "near",       "far from"),
    ("person", "bus",     "near",       "far from"),
    ("person", "bus",     "left of",    "right of"),
    ("person", "chair",   "on",         "under"),
    ("dog",    "couch",   "on",         "under"),
    ("dog",    "couch",   "on top of",  "inside"),
    ("cup",    "table",   "on",         "under"),
    ("bottle", "table",   "on",         "inside"),
]

results_table = []

for subj_q, obj_q, true_rel, corrupt_rel in CORRUPTION_TESTS:
    # Find an image where both objects are detected
    found_img = None
    for name, data in all_detections.items():
        f = data["found"]
        if subj_q in f and obj_q in f:
            found_img = (name, f[subj_q][0]["box"], f[obj_q][0]["box"])
            break

    if not found_img:
        results_table.append({
            "test": f"({subj_q}, {true_rel}, {obj_q}) → {corrupt_rel}",
            "image": "—",
            "true_verdict": "—",
            "corrupt_verdict": "—",
            "detected": "NO OBJECTS",
        })
        continue

    img_name, subj_box, obj_box = found_img

    true_result    = verify_spatial(true_rel,    subj_box, obj_box)
    corrupt_result = verify_spatial(corrupt_rel, subj_box, obj_box)

    true_v = true_result["verdict"]
    corrupt_v = corrupt_result["verdict"]

    # Detection success = true relation is True AND corrupted is False
    if true_v == True and corrupt_v == False:
        detected = "✅ YES"
    elif true_v == True and corrupt_v == True:
        detected = "⚠️  BOTH TRUE"
    elif true_v == False:
        detected = "❌ TRUE FAILED"
    elif corrupt_v is None or true_v is None:
        detected = "❓ NO RULE"
    else:
        detected = "❌ MISSED"

    results_table.append({
        "test": f"({subj_q}, {true_rel}, {obj_q}) → {corrupt_rel}",
        "image": img_name,
        "true_verdict": f"{true_v} — {true_result['evidence'][:40]}",
        "corrupt_verdict": f"{corrupt_v} — {corrupt_result['evidence'][:40]}",
        "detected": detected,
    })

# Print results table
print(f"\n{'Test':<50s} {'Image':<16s} {'Detected?'}")
print("-" * 80)
for r in results_table:
    print(f"{r['test']:<50s} {r['image']:<16s} {r['detected']}")

print(f"\n{'='*70}")
detected_count = sum(1 for r in results_table if "YES" in r["detected"])
total = len(results_table)
no_obj = sum(1 for r in results_table if "NO OBJECTS" in r["detected"])
print(f"Detected: {detected_count}/{total-no_obj} (excluding {no_obj} with no detections)")
print(f"{'='*70}")

# Also print detailed verdicts for debugging
print("\nDetailed verdicts:")
for r in results_table:
    print(f"\n  {r['test']}")
    print(f"    True:    {r['true_verdict']}")
    print(f"    Corrupt: {r['corrupt_verdict']}")
